# 03b — Full-Trajectory Conflict Evaluation (All Checkpoints, LLM Judge)
## Context-Parametric Inversion Study

**Purpose:** evaluate **every** saved Phase-2 checkpoint (currently every
50 steps up to step 1500) on the full conflict-evaluation pipeline in
`eval.py`, this time with the **real Layer-2 LLM judge** resolving AMBIG
items — unlike `00_Pilot_Run.ipynb`, which used the no-judge
`ordered`-method fallback, and unlike `03_Trajectory_Evaluation.ipynb`,
which only deep-evaluates 4 hand-picked tagged checkpoints (base/peak/
mid-decline/final). This notebook is the dense, judge-backed version of
that trajectory: one evaluated point per saved checkpoint, all using the
same rigor (full item set, default distractor count, real judge).

**Relationship to other notebooks:**
- Reuses `eval.py` exactly as-is (`load_items`, `build_distractor_pool`,
  `load_model`, `evaluate_checkpoint`, `make_judge`) — no reimplemented
  classification logic, same as Notebooks 0 and 3.
- Reads checkpoints from the **real Phase-2 run** (`CHECKPOINT_DIR` under
  `RUN_TAG`, as written by `02_LoRA_Finetuning.ipynb`), not the pilot's
  checkpoint directory.
- Produces a trajectory plot in the same two-panel style as the pilot's
  Section 12 (R_ctx / R_par on the left, filter yield on the right), with
  one addition: since a real judge is used this time, R_ctx is plotted
  with its bootstrap 95% CI, and the pre-judge (`logprob`-only, AMBIG
  left unresolved) rate is shown alongside it so the judge's effect is
  visible rather than hidden.

**Before running:** make sure `OPENAI_API_KEY` is available (Colab
Secrets or environment) — without it, this notebook will warn and fall
back to the pilot-style no-judge resolution, which defeats the purpose of
re-running this evaluation.

## Section 1 — Setup

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU. Runtime → Change runtime type → A100 GPU.')
print(f'GPU: {r.stdout.strip()}')
print('bf16 base model (required by eval.py) needs more headroom than 4-bit — A100 recommended.')


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
bf16 base model (required by eval.py) needs more headroom than 4-bit — A100 recommended.


In [ ]:
rm -rf ~/.cache/huggingface/xet

In [ ]:
rm -rf ~/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B

In [1]:
# No bitsandbytes: eval.py requires bf16 (not 4-bit) for valid log-prob comparison.
# %pip uninstall -y -q peft torchao
%pip install -q -U \
    transformers \
    peft \
    torchao \
    datasets \
    scipy \
    matplotlib \
    openai
print('Dependencies installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 221.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 181.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 233.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 65.7 MB/s eta 0:00:00
Dependencies installed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/context-parametric-inversion-research'
os.makedirs(f'{PROJECT}/figures', exist_ok=True)
print(f'Project root: {PROJECT}')


Mounted at /content/drive
Project root: /content/drive/MyDrive/context-parametric-inversion-research


In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in to Hugging Face.')
except Exception:
    import getpass
    login(token=getpass.getpass('HF token: '))

Logged in to Hugging Face.


In [ ]:
# OpenAI API key for the Layer 2 judge -- REQUIRED for this notebook to do
# what it's for (this is the judge-backed re-evaluation; without a key it
# silently degrades to the same fallback the pilot already used).
import os
try:
    from google.colab import userdata
    key = userdata.get('OPENAI_API_KEY')
    if key:
        os.environ['OPENAI_API_KEY'] = key
        print('OPENAI_API_KEY set from Colab Secrets.')
    else:
        print('WARNING: No OPENAI_API_KEY in Secrets. USE_LLM_JUDGE will fail to '
              'initialise below and this run will fall back to no-judge AMBIG '
              'resolution -- the same approximation the pilot already used.')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key (for the Layer 2 judge): ')


OPENAI_API_KEY set from Colab Secrets.


## Section 2 — Configuration

**`MODEL_NAME`/`SEED` must match the Phase 2 run** (`RUN_TAG` below must
resolve to the same checkpoint directory Notebook 2 wrote to). **Verify
`EVAL_PY_PATH` and `DATASET_PATH`** match your actual Drive layout.

`CHECKPOINT_STEP_INTERVAL` / `CHECKPOINT_MAX_STEP` restrict this run to
the checkpoints you actually have right now (every 50 steps, up to 1500).
Bump `CHECKPOINT_MAX_STEP` later and re-run this notebook -- it's
resumable, so already-evaluated steps are skipped.

In [ ]:
# ═══════════════════════════════════════
# USER SETTINGS
# ═══════════════════════════════════════
MODEL_NAME = 'meta-llama/Llama-3.1-8B'   # must match the Phase 2 run
SEED       = 0                            # must match the Phase 2 run

EVAL_PY_PATH = '/content/drive/MyDrive/context-parametric-inversion-research/evaluation/eval.py'
DATASET_PATH = '/content/drive/MyDrive/context-parametric-inversion-research/dataset/conflict_eval_unified.json'

# Which checkpoints to evaluate this run. Set CHECKPOINT_MAX_STEP higher
# (and re-run) as training progresses -- already-evaluated steps are
# skipped automatically via the resumable trajectory log.
CHECKPOINT_STEP_INTERVAL = 50
CHECKPOINT_MAX_STEP      = 1500
INCLUDE_FINAL_CHECKPOINT = False   # 'final' checkpoint, if present -- usually beyond MAX_STEP

# Evaluation set size. None = full conflict item set (most rigorous, slowest
# and most expensive against the judge). Set an int (e.g. 80) to subsample
# for a faster/cheaper first pass across all checkpoints.
EVAL_SAMPLE_SIZE = None

# This is the whole point of this notebook vs. the pilot: resolve AMBIG
# items with the real judge instead of the ordered-method fallback.
USE_LLM_JUDGE = True

# ═══════════════════════════════════════
# DERIVED SETTINGS
# ═══════════════════════════════════════
import os, torch, random, json

MODEL_ID_MAP = {
    'meta-llama/Llama-3.1-8B':   'llama31_8b',
    'mistralai/Mistral-7B-v0.3': 'mistral_7b',
}
MODEL_ID = MODEL_ID_MAP.get(MODEL_NAME, MODEL_NAME.split('/')[-1].lower())
RUN_TAG  = f'{MODEL_ID}_r128_seed{SEED}'

CHECKPOINT_DIR = f'{PROJECT}/sft/{RUN_TAG}'
RESULTS_DIR    = f'{PROJECT}/results/{RUN_TAG}'
FIG_DIR        = f'{PROJECT}/figures'
EVAL_JSON_DIR  = f'{RESULTS_DIR}/eval_py_checkpoints_full_judge'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EVAL_JSON_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

TRAJ_LOG_PATH = f'{RESULTS_DIR}/full_trajectory_llm_judge.jsonl'

print(f'Run tag          : {RUN_TAG}')
print(f'Checkpoint dir   : {CHECKPOINT_DIR}')
print(f'Results dir      : {RESULTS_DIR}')
print(f'Trajectory log   : {TRAJ_LOG_PATH}')
print(f'eval.py path     : {EVAL_PY_PATH}   (exists: {os.path.exists(EVAL_PY_PATH)})')
print(f'Dataset path     : {DATASET_PATH}   (exists: {os.path.exists(DATASET_PATH)})')

assert os.path.exists(EVAL_PY_PATH), 'eval.py not found at EVAL_PY_PATH -- fix the path above.'
assert os.path.exists(DATASET_PATH), 'Dataset not found at DATASET_PATH -- fix the path above.'
assert os.path.isdir(CHECKPOINT_DIR), (
    f'{CHECKPOINT_DIR} not found -- confirm MODEL_NAME/SEED match the Phase 2 run.')


Run tag          : llama31_8b_r128_seed0
Checkpoint dir   : /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0
Results dir      : /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0
Trajectory log   : /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0/full_trajectory_llm_judge.jsonl
eval.py path     : /content/drive/MyDrive/context-parametric-inversion-research/evaluation/eval.py   (exists: True)
Dataset path     : /content/drive/MyDrive/context-parametric-inversion-research/dataset/conflict_eval_unified.json   (exists: True)


## Section 3 — Import `eval.py`

In [ ]:
import sys
eval_py_dir = os.path.dirname(EVAL_PY_PATH)
if eval_py_dir not in sys.path:
    sys.path.insert(0, eval_py_dir)

import eval as cpi_eval
print(f'Imported eval.py version {cpi_eval.__version__}')
print(f'TAU={cpi_eval.TAU}  K_DISTRACTORS={cpi_eval.K_DISTRACTORS}  '
      f'MAX_NEW_TOKENS={cpi_eval.MAX_NEW_TOKENS}')
cpi_eval.run_offline_self_tests()


Imported eval.py version 1.0.0
TAU=1.0  K_DISTRACTORS=5  MAX_NEW_TOKENS=24
classifier regression test: PASS
scorer indexing test (incl. EOS-strip): PASS
aggregation test: PASS
McNemar test (incl. paired-flags helper): PASS

ALL OFFLINE SELF-TESTS PASS


## Section 4 — Initialise the LLM Judge

Unlike the pilot (which had no judge, by design) and unlike Notebook 3's
Stage A scan (also judge-less, for speed), **this notebook's whole reason
for existing is to use the real judge.** If initialisation fails, that's
surfaced loudly rather than silently degrading.

In [ ]:
judge_fn = None
if USE_LLM_JUDGE:
    try:
        judge_fn = cpi_eval.make_judge()
        print('LLM judge initialised -- AMBIG items will be resolved by the real judge.')
    except RuntimeError as e:
        print(f'Could not initialise judge: {e}')
        print('Falling back to no-judge AMBIG resolution for this run '
              '(same approximation the pilot already used -- consider fixing '
              'OPENAI_API_KEY and re-running before trusting these numbers).')
        USE_LLM_JUDGE = False
else:
    print('USE_LLM_JUDGE=False -- AMBIG items will be left unresolved (no fallback applied).')


LLM judge initialised -- AMBIG items will be resolved by the real judge.


## Section 5 — Load Conflict Evaluation Data

In [ ]:
all_items = cpi_eval.load_items(DATASET_PATH)
distractor_pool = cpi_eval.build_distractor_pool(all_items)

random.seed(42)
if EVAL_SAMPLE_SIZE is None:
    eval_items = all_items
else:
    eval_items = random.sample(all_items, min(EVAL_SAMPLE_SIZE, len(all_items)))

print(f'Full item set (for distractor pool): {len(all_items)}')
print(f'Evaluated per checkpoint this run  : {len(eval_items)}')


Loaded 417 total | dropped 5 flagged item(s) | 412 usable
Full item set (for distractor pool): 412
Evaluated per checkpoint this run  : 412


## Section 6 — Load Base Model in bf16

Via `eval.py`'s own `load_model()` -- required precision for valid
cross-checkpoint log-prob comparison. This model stays loaded; each
checkpoint's LoRA adapter is attached/swapped/removed on top of it, same
pattern as the pilot's Section 10-11.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
base_model.eval()
print("base model loaded")

# Aliased to the names the rest of this notebook expects
eval_base_model = base_model
eval_tokenizer  = tok


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

base model loaded


## Section 7 — Discover Checkpoints to Evaluate

Restricts to steps that are multiples of `CHECKPOINT_STEP_INTERVAL`, up
to `CHECKPOINT_MAX_STEP` -- i.e. exactly the checkpoints you have right
now. Re-run this notebook later with a higher `CHECKPOINT_MAX_STEP` to
pick up newer checkpoints; already-evaluated steps are skipped.

In [ ]:
import glob

all_ckpt_dirs = sorted(
    glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda p: int(p.split('-')[-1])
)

def step_of(p):
    return int(p.split('-')[-1])

ckpt_dirs = [p for p in all_ckpt_dirs
             if step_of(p) % CHECKPOINT_STEP_INTERVAL == 0
             and step_of(p) <= CHECKPOINT_MAX_STEP]

if INCLUDE_FINAL_CHECKPOINT and os.path.isdir(f'{CHECKPOINT_DIR}/final'):
    ckpt_dirs.append(f'{CHECKPOINT_DIR}/final')

print(f'Found {len(all_ckpt_dirs)} total checkpoint(s) in {CHECKPOINT_DIR}')
suffix = ' (+final)' if INCLUDE_FINAL_CHECKPOINT else ''
print(f'Selected {len(ckpt_dirs)} checkpoint(s) matching every '
      f'{CHECKPOINT_STEP_INTERVAL} steps up to {CHECKPOINT_MAX_STEP}{suffix}:')
for p in ckpt_dirs:
    print(f'  {p}')

if not ckpt_dirs:
    raise RuntimeError(
        'No checkpoints matched -- check CHECKPOINT_STEP_INTERVAL/CHECKPOINT_MAX_STEP '
        'against what actually exists in CHECKPOINT_DIR (printed above).')

est_seconds_per_item = 3.0   # rough baseline; judge escalations add extra latency
n_points = len(ckpt_dirs) + 1  # +1 for the true base model
est_total_minutes = (len(eval_items) * n_points * est_seconds_per_item) / 60
print(f'\nRough time estimate: ~{est_total_minutes:.0f} minutes '
      f'({len(eval_items)} items x {n_points} points x ~{est_seconds_per_item}s/item, '
      f'before judge latency on escalated/AMBIG items).')
print('Judge calls add real wall-clock time and API cost on top of this -- the exact amount '
      'depends on how often Layer 1 lands in AMBIG, i.e. the escalation_rate you will see below.')


Found 30 total checkpoint(s) in /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0
Selected 30 checkpoint(s) matching every 50 steps up to 1500:
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-50
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-100
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-150
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-200
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-250
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-300
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-350
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/che

## Section 8 — Bind the Evaluation Call

Same pattern as the pilot: bind the static arguments once with
`functools.partial`, then call with just `(model, tokenizer,
checkpoint_id, global_step)` per checkpoint. `judge_fn_=judge_fn` is the
one substantive difference from the pilot's binding.

In [ ]:
import functools

eval_ckpt = functools.partial(
    cpi_eval.evaluate_checkpoint,
    items=eval_items, distractor_pool=distractor_pool,
    output_path=EVAL_JSON_DIR, methods=('logprob', 'ordered'),
    judge_fn_=judge_fn, verbose=False,
)
print('Evaluation call bound.')

Evaluation call bound.


## Section 9 — Resumable Trajectory Log

Loads any previously-computed points for this run so re-running the
notebook (e.g. after raising `CHECKPOINT_MAX_STEP`) doesn't re-evaluate
(and re-charge the judge for) checkpoints already done.

In [ ]:
trajectory = []
evaluated_steps = set()
if os.path.exists(TRAJ_LOG_PATH):
    with open(TRAJ_LOG_PATH) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                trajectory.append(row)
                evaluated_steps.add(row['global_step'])
    print(f'Loaded {len(trajectory)} previously-evaluated checkpoint(s).')
else:
    print('No previous trajectory log found -- starting fresh.')


No previous trajectory log found -- starting fresh.


## Section 10 — Rate Extraction Helper

Unlike the pilot's `pilot_final_rate` (which had to substitute the
`ordered` label for AMBIG items because no judge was available),
`res['summary']['aggregate']['final']` here is **already** judge-resolved
by `eval.py` itself (that's what passing `judge_fn_` to
`evaluate_checkpoint` does). This helper just pulls out the numbers we
want to plot, plus the pre-judge `logprob`-only rate for comparison.

In [ ]:
def extract_rates(res):
    agg = res['aggregate']
    final = agg.get('final', {})
    logp  = agg.get('logprob', {})
    return {
        'n_passed':            agg.get('n_passed', 0),
        'filter_yield':        res['filter_yield'],
        'R_ctx_final':         final.get('R_ctx', {}).get('mean'),
        'R_ctx_final_ci':      final.get('R_ctx', {}).get('ci'),
        'R_par_final':         final.get('R_par', {}).get('mean'),
        'R_other_final':       final.get('R_other', {}).get('mean'),
        'R_ctx_raw_logprob':   logp.get('R_ctx', {}).get('mean'),
        'R_ambiguous_raw':     logp.get('R_ambiguous', {}).get('mean'),
        'escalation_rate':     agg.get('escalation_rate'),
        'layer2_failed_rate':  agg.get('layer2_failed_rate'),
    }

print('Rate extraction helper defined.')


Rate extraction helper defined.


## Section 11 — Evaluate the True Base Model (Step 0)

Same as the pilot: the un-adapted base model anchors the trajectory at
step 0.

In [ ]:
import time

if 0 not in evaluated_steps:
    print('Evaluating TRUE BASE model (step 0, no LoRA adapter) ...')
    t0 = time.time()
    res = eval_ckpt(eval_base_model, eval_tokenizer,
                    checkpoint_id='full_base', global_step=0, use_ct=True)
    elapsed = time.time() - t0
    rates = extract_rates(res)
    row = {'global_step': 0, 'checkpoint_id': 'full_base', **rates,
           'eval_seconds': round(elapsed, 1)}
    trajectory.append(row)
    evaluated_steps.add(0)
    with open(TRAJ_LOG_PATH, 'a') as f: f.write(json.dumps(row) + '\n')
    print(f'  R_ctx (final, judge-resolved) = {rates["R_ctx_final"]:.3f}  '
          f'({elapsed:.0f}s for {len(eval_items)} items)')
else:
    print('Base model (step 0) already evaluated -- skipping.')


Evaluating TRUE BASE model (step 0, no LoRA adapter) ...
Wrote /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0/eval_py_checkpoints_full_judge/checkpoint_full_base_metrics.json
  R_ctx (final, judge-resolved) = 0.729  (181s for 412 items)


## Section 12 — Stage Checkpoints Locally

Copies each not-yet-evaluated checkpoint from Drive to local Colab disk
first (same pattern as the pilot) -- reading LoRA adapter weights
repeatedly straight off Drive is the slow part, not the forward passes.

In [ ]:
import os
import re
import shutil
import time

DRIVE_CKPT_ROOT = '/content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0'
LOCAL_CKPT_ROOT = '/content/ckpt_local_full'
os.makedirs(LOCAL_CKPT_ROOT, exist_ok=True)

def step_of(path):
    """Extract the step number from a 'checkpoint-N' folder name."""
    m = re.search(r'checkpoint-(\d+)$', path.rstrip('/'))
    if m:
        return int(m.group(1))
    raise ValueError(f'Could not parse step from: {path}')

# Discover all checkpoint-* dirs under the Drive root, sorted by step
ckpt_dirs = sorted(
    (os.path.join(DRIVE_CKPT_ROOT, d) for d in os.listdir(DRIVE_CKPT_ROOT)
     if d.startswith('checkpoint-') and os.path.isdir(os.path.join(DRIVE_CKPT_ROOT, d))),
    key=step_of
)

print(f'Found {len(ckpt_dirs)} checkpoint dirs on Drive:')
for d in ckpt_dirs:
    print(f'  {d}  (step {step_of(d)})')

local_map = {}   # step -> local dir
t0 = time.time()
for src in ckpt_dirs:
    step = step_of(src)
    if step in evaluated_steps:
        continue
    dst = os.path.join(LOCAL_CKPT_ROOT, os.path.basename(src))
    if not os.path.isdir(dst):
        shutil.copytree(src, dst)
    f = os.path.join(dst, 'adapter_model.safetensors')
    sz = os.path.getsize(f) if os.path.exists(f) else 0
    assert sz > 100_000_000, f'{dst} incomplete ({sz} bytes)'
    local_map[step] = dst
    print(f'  staged {os.path.basename(src):20s} {sz/1e9:.3f} GB', flush=True)

print(f'\nPrefetched {len(local_map)} checkpoint(s) needing evaluation in {time.time()-t0:.0f}s')
if not local_map:
    print('Nothing new to stage -- all selected checkpoints already evaluated.')

Found 30 checkpoint dirs on Drive:
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-50  (step 50)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-100  (step 100)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-150  (step 150)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-200  (step 200)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-250  (step 250)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-300  (step 300)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-350  (step 350)
  /content/drive/MyDrive/context-parametric-inversion-research/sft/llama31_8b_r128_seed0/checkpoint-400  (step 400)
  /content/drive/MyDrive/context-parame

## Section 13 — Per-Checkpoint Evaluation

Attaches each checkpoint's LoRA adapter on top of the already-loaded base
model in turn (no reload), evaluates with the judge live, logs the row,
then detaches the adapter before moving to the next one. Resumable: a
re-run skips anything already in `TRAJ_LOG_PATH`.

In [ ]:
print(local_map)
local_map={
     100: '/content/ckpt_local_full/checkpoint-100', 150: '/content/ckpt_local_full/checkpoint-150', 200: '/content/ckpt_local_full/checkpoint-200', 250: '/content/ckpt_local_full/checkpoint-250', 300: '/content/ckpt_local_full/checkpoint-300', 350: '/content/ckpt_local_full/checkpoint-350', 400: '/content/ckpt_local_full/checkpoint-400', 450: '/content/ckpt_local_full/checkpoint-450', 500: '/content/ckpt_local_full/checkpoint-500', 550: '/content/ckpt_local_full/checkpoint-550', 600: '/content/ckpt_local_full/checkpoint-600', 650: '/content/ckpt_local_full/checkpoint-650', 700: '/content/ckpt_local_full/checkpoint-700', 750: '/content/ckpt_local_full/checkpoint-750', 800: '/content/ckpt_local_full/checkpoint-800', 850: '/content/ckpt_local_full/checkpoint-850', 900: '/content/ckpt_local_full/checkpoint-900', 950: '/content/ckpt_local_full/checkpoint-950', 1000: '/content/ckpt_local_full/checkpoint-1000', 1050: '/content/ckpt_local_full/checkpoint-1050', 1100: '/content/ckpt_local_full/checkpoint-1100', 1150: '/content/ckpt_local_full/checkpoint-1150', 1200: '/content/ckpt_local_full/checkpoint-1200', 1250: '/content/ckpt_local_full/checkpoint-1250', 1300: '/content/ckpt_local_full/checkpoint-1300', 1350: '/content/ckpt_local_full/checkpoint-1350', 1400: '/content/ckpt_local_full/checkpoint-1400', 1450: '/content/ckpt_local_full/checkpoint-1450', 1500: '/content/ckpt_local_full/checkpoint-1500'
}
print(f"updated local map : {local_map}")

{100: '/content/ckpt_local_full/checkpoint-100', 150: '/content/ckpt_local_full/checkpoint-150', 200: '/content/ckpt_local_full/checkpoint-200', 250: '/content/ckpt_local_full/checkpoint-250', 300: '/content/ckpt_local_full/checkpoint-300', 350: '/content/ckpt_local_full/checkpoint-350', 400: '/content/ckpt_local_full/checkpoint-400', 450: '/content/ckpt_local_full/checkpoint-450', 500: '/content/ckpt_local_full/checkpoint-500', 550: '/content/ckpt_local_full/checkpoint-550', 600: '/content/ckpt_local_full/checkpoint-600', 650: '/content/ckpt_local_full/checkpoint-650', 700: '/content/ckpt_local_full/checkpoint-700', 750: '/content/ckpt_local_full/checkpoint-750', 800: '/content/ckpt_local_full/checkpoint-800', 850: '/content/ckpt_local_full/checkpoint-850', 900: '/content/ckpt_local_full/checkpoint-900', 950: '/content/ckpt_local_full/checkpoint-950', 1000: '/content/ckpt_local_full/checkpoint-1000', 1050: '/content/ckpt_local_full/checkpoint-1050', 1100: '/content/ckpt_local_full/che

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.7 MB/s eta 0:00:00


In [ ]:
import gc

peft_model = None
for step in sorted(local_map):
    local_dir = local_map[step]
    tag = 'final' if local_dir.endswith('/final') else f'checkpoint-{step}'
    print(f'Evaluating {tag} (step {step}) ...', flush=True)
    t0 = time.time()

    adapter_name = f'ckpt_{step}'
    if peft_model is None:
        peft_model = PeftModel.from_pretrained(eval_base_model, local_dir,
                                               adapter_name=adapter_name, is_trainable=False)
    else:
        peft_model.load_adapter(local_dir, adapter_name=adapter_name)
    peft_model.set_adapter(adapter_name)
    peft_model.config.use_cache = True
    peft_model.eval()

    res = eval_ckpt(peft_model, eval_tokenizer,
                    checkpoint_id=f'full_step{step}', global_step=step, use_ct=True)
    elapsed = time.time() - t0
    print(f'  done in {elapsed:.0f}s', flush=True)

    rates = extract_rates(res)
    row = {'global_step': step, 'checkpoint_id': tag, **rates,
           'eval_seconds': round(elapsed, 1)}
    trajectory.append(row)
    evaluated_steps.add(step)
    with open(TRAJ_LOG_PATH, 'a') as f: f.write(json.dumps(row) + '\n')
    print(f'  R_ctx(final)={rates["R_ctx_final"]:.3f}  '
          f'R_ctx(raw logprob)={rates["R_ctx_raw_logprob"]}  '
          f'escalation_rate={rates["escalation_rate"]}  '
          f'filter_yield={rates["filter_yield"]:.2f}')

    peft_model.delete_adapter(adapter_name)
    torch.cuda.empty_cache()

del peft_model; gc.collect(); torch.cuda.empty_cache()
trajectory.sort(key=lambda r: r['global_step'])
print(f'\nEvaluation complete. {len(trajectory)} point(s) in the trajectory.')


Evaluating checkpoint-100 (step 100) ...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Wrote /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0/eval_py_checkpoints_full_judge/checkpoint_full_step100_metrics.json
  done in 272s
  R_ctx(final)=0.800  R_ctx(raw logprob)=0.52  escalation_rate=0.461  filter_yield=0.98
Evaluating checkpoint-150 (step 150) ...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1596: UserWarning: Adapter ckpt_100 was active which is now deleted. Setting active adapter to ckpt_50.
  warnings.warn(


Wrote /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0/eval_py_checkpoints_full_judge/checkpoint_full_step150_metrics.json
  done in 240s
  R_ctx(final)=0.855  R_ctx(raw logprob)=0.68  escalation_rate=0.303  filter_yield=0.98
Evaluating checkpoint-200 (step 200) ...


...

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1596: UserWarning: Adapter ckpt_1400 was active which is now deleted. Setting active adapter to ckpt_50.
  warnings.warn(


Wrote /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0/eval_py_checkpoints_full_judge/checkpoint_full_step1450_metrics.json
  done in 404s
  R_ctx(final)=0.928  R_ctx(raw logprob)=0.632  escalation_rate=0.358  filter_yield=0.98
Evaluating checkpoint-1500 (step 1500) ...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1596: UserWarning: Adapter ckpt_1450 was active which is now deleted. Setting active adapter to ckpt_50.
  warnings.warn(


Wrote /content/drive/MyDrive/context-parametric-inversion-research/results/llama31_8b_r128_seed0/eval_py_checkpoints_full_judge/checkpoint_full_step1500_metrics.json
  done in 363s
  R_ctx(final)=0.923  R_ctx(raw logprob)=0.637  escalation_rate=0.356  filter_yield=0.98

Evaluation complete. 30 point(s) in the trajectory.

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1596: UserWarning: Adapter ckpt_1500 was active which is now deleted. Setting active adapter to ckpt_50.
  warnings.warn(


## Section 14 — Plot the Full Trajectory

Same two-panel layout as the pilot's Section 12: left panel is
R_ctx/R_par, right panel is filter yield. The difference from the pilot
plot: R_ctx here is the **judge-resolved** rate (with 95% bootstrap CI as
an error band) rather than a fallback approximation, plotted alongside
the pre-judge `logprob`-only rate so you can see how much the judge
actually moved the curve.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

steps           = [r['global_step'] for r in trajectory]
r_ctx_final     = [r['R_ctx_final'] for r in trajectory]
r_ctx_raw       = [r['R_ctx_raw_logprob'] for r in trajectory]
r_par_final     = [r['R_par_final'] for r in trajectory]
filter_yield    = [r['filter_yield'] for r in trajectory]
ci_lo = [r['R_ctx_final_ci'][0] if r['R_ctx_final_ci'] else r['R_ctx_final'] for r in trajectory]
ci_hi = [r['R_ctx_final_ci'][1] if r['R_ctx_final_ci'] else r['R_ctx_final'] for r in trajectory]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

judge_label = 'judge-resolved' if USE_LLM_JUDGE else 'no-judge fallback'
ax1 = axes[0]
ax1.fill_between(steps, ci_lo, ci_hi, color='tab:blue', alpha=0.15, label='R_ctx 95% CI')
ax1.plot(steps, r_ctx_final, 'b-o', markersize=4, label=f'R_ctx (final, {judge_label})')
ax1.plot(steps, r_ctx_raw,   'b--s', markersize=3, alpha=0.5, label='R_ctx (raw, logprob-only, pre-judge)')
ax1.plot(steps, r_par_final, 'r--^', markersize=3, label='R_par (final)')
ax1.set_xlabel('Training step'); ax1.set_ylabel('Rate')
ax1.set_title(f'Full Checkpoint Trajectory — {RUN_TAG}  (n={len(eval_items)} items/checkpoint)')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(steps, filter_yield, 'g-o', markersize=4)
ax2.set_xlabel('Training step'); ax2.set_ylabel('Filter yield')
ax2.set_title('Genuine-Conflict Filter Yield (should stay stable)')
ax2.set_ylim(0, 1.05); ax2.grid(alpha=0.3)

plt.tight_layout()
fig_path = f'{FIG_DIR}/{RUN_TAG}_full_trajectory_llm_judge.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {fig_path}')


## Section 15 — Quick Summary Stats

Base / peak / current-final R_ctx, for a quick read of the shape without
re-deriving checkpoint-density recommendations (that calibration step
already happened in the pilot notebook -- this notebook is about
measuring the real run, not planning it).

In [ ]:
r_ctx_arr = np.array(r_ctx_final)
base_r    = r_ctx_final[0]
peak_idx  = int(np.argmax(r_ctx_arr))
peak_step = steps[peak_idx]
last_r    = r_ctx_final[-1]
last_step = steps[-1]

print(f'Base  R_ctx  : {base_r:.3f}  (step 0)')
print(f'Peak  R_ctx  : {r_ctx_final[peak_idx]:.3f}  at step {peak_step}  '
      f'CI [{ci_lo[peak_idx]:.3f}, {ci_hi[peak_idx]:.3f}]')
print(f'Latest R_ctx : {last_r:.3f}  at step {last_step} (most recent checkpoint evaluated)')
print(f'Change base to peak  : {100*(r_ctx_final[peak_idx]-base_r):+.1f}pp')
print(f'Change peak to latest: {100*(last_r-r_ctx_final[peak_idx]):+.1f}pp')
print(f'\nJudge used this run : {USE_LLM_JUDGE}')
print('Note: this is a running trajectory, not a final result -- re-run with a higher '
      'CHECKPOINT_MAX_STEP as training progresses past step 1500; already-evaluated '
      'points are skipped automatically.')


## Section 16 — Save Trajectory Summary

In [ ]:
summary_out = {
    'run_tag': RUN_TAG,
    'model_name': MODEL_NAME,
    'checkpoint_dir': CHECKPOINT_DIR,
    'checkpoint_step_interval': CHECKPOINT_STEP_INTERVAL,
    'checkpoint_max_step': CHECKPOINT_MAX_STEP,
    'n_items_per_checkpoint': len(eval_items),
    'llm_judge_used': USE_LLM_JUDGE,
    'eval_py_version': cpi_eval.__version__,
    'base_R_ctx': base_r,
    'peak_R_ctx': r_ctx_final[peak_idx],
    'peak_step': peak_step,
    'latest_R_ctx': last_r,
    'latest_step': last_step,
    'trajectory': trajectory,
}
summary_path = f'{RESULTS_DIR}/full_trajectory_llm_judge_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary_out, f, indent=2)
print(f'Saved → {summary_path}')


---
## Next Steps

1. Review the plot in Section 14 -- specifically how far the judge-resolved
   curve (`R_ctx_final`) diverges from the pre-judge `logprob`-only curve;
   a big gap means AMBIG resolution mattered a lot for your reported numbers.
2. As training progresses past step 1500, raise `CHECKPOINT_MAX_STEP` in
   Section 2 and re-run this notebook -- already-evaluated checkpoints are
   skipped, so only the new ones get evaluated (and judged).
3. `eval.py`'s own per-checkpoint JSON files (full per-item detail, one
   file per step) are under `EVAL_JSON_DIR` if you need to audit any
   individual item's classification, escalation, or judge verdict.
4. If you want the smaller, statistically-tested (McNemar) comparison
   across base/peak/mid-decline/final specifically, that's what
   `03_Trajectory_Evaluation.ipynb` Sections 10-12 already do -- this
   notebook is the dense, every-checkpoint complement to that, not a
   replacement for it.